# **ginger**

In [1]:
import sys

if "google.colab" in sys.modules:
    import getpass

    token = getpass.getpass()

    !git clone https://{token}@github.com/Mango-Cats/ginger.git
    %cd ginger

    try:
        %load_ext autoreload
        %autoreload 2
    except Exception as e:
        print("autoreload not available:", e)

    !pip install -e . --no-deps

fatal: destination path 'ginger' already exists and is not an empty directory.
/content/ginger
autoreload not available: No module named 'imp'
Obtaining file:///content/ginger
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ginger (pyproject.toml) ... done
  Created wheel for ginger: filename=ginger-0.1.0-0.editable-py3-none-any.whl size=5480 sha256=d54adda81a1077f8b65b1ba245a9fb561899577f351821092575e366ee7c69fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-fcklu13v/wheels/4a/45/44/d3f200324dc86253cc07b97b9015a378520aae0b30f78eb83b
Successfully built ginger
  Attempting uninstall: ginger
    Found existing installation: ginger 0.1.0
    Uninstalling ginger-0.1.0:
      Successfully uninstalled ginger-0.1.0


In [2]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

2.10.0+cu128
cuda


## **Data Loading & Preprocessing**

In [3]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [4]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [5]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
import torch.nn as nn

In [7]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


## **Training**

In [10]:
from src.trainer import evaluate, train_epoch

ImportError: cannot import name 'train_epoch' from 'src.trainer' (/content/ginger/src/trainer/__init__.py)

In [13]:
PATIENCE = 7

no_improve = 0
best_val_loss = float('inf')
best_model_path = "best_g2p_model.pt"

train_losses = []
val_losses = []

print(f"\nStarting training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model=model, dataloader=train_loader, criterion=CRITERION, optimizer=OPTIMIZER, scheduler=SCHEDULER, device=DEVICE)
    val_loss = evaluate(model, val_loader, CRITERION, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        status = "(New Best! Model saved)"
    else:
        no_improve += 1
        status = f"(No improvement for {no_improve} epochs)"
    
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} {status}")
    
    if no_improve >= PATIENCE:
        print(f"\nStopping. Validation loss hasn't improved for {PATIENCE} epochs.")
        print(f"Loading best model weights from epoch {epoch + 1 - PATIENCE}...")
        model.load_state_dict(torch.load(best_model_path))
        break


Starting training for 100 epochs...



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


ValueError: Tried to step 22801 times. The specified number of total steps is 22800

## **Inference**

In [10]:
import random
from src.architecture.inference import greedy_decode
from src.evaluation.metrics import compute_metrics

In [11]:
test_pairs = test_loader.dataset.data_pairs
predictions = []
references = []

for i, (graphemes, true_phonemes) in enumerate(test_pairs):
    word = ''.join(graphemes)
    
    predicted_phonemes, _ = greedy_decode(model, word, grapheme_vocab, phoneme_vocab, DEVICE)
    
    predictions.append(predicted_phonemes)
    references.append(true_phonemes)

In [12]:
per, wer = compute_metrics(predictions, references)
total_words = len(test_pairs)
print("\nMetrics:")
print(f"Total Words Tested: {total_words}")
print(f"Phoneme Error Rate (PER): {per:.2f}%")
print(f"Word Error Rate (WER):    {wer:.2f}%")

print("\nSample Predictions:")
sample_indices = random.sample(range(total_words), 5)
for idx in sample_indices:
    grap = ''.join(test_pairs[idx][0])
    print(f"\nWord:     {grap.upper()}")
    print(f"Expected: {' '.join(references[idx])}")
    print(f"Output:   {' '.join(predictions[idx])}")


Metrics:
Total Words Tested: 1824
Phoneme Error Rate (PER): 3.00%
Word Error Rate (WER):    15.84%

Sample Predictions:

Word:     CARVAJAL
Expected: k a ɾ v a h a l
Output:   k a ɾ b a h a l

Word:     ARAKIBOT
Expected: ʔ a ɾ a k i b o t
Output:   ʔ a ɾ a k i b o t

Word:     ORARYO
Expected: ʔ o ɾ a ɾ j o
Output:   ʔ o ɾ a ɾ j o

Word:     INANG
Expected: ʔ i n a ŋ
Output:   ʔ i n a ŋ

Word:     TIO
Expected: t j o
Output:   t i o ʔ o
